# 00 — Environment Setup (Colab)

Run this notebook once at the start of every Colab session to:
1. Verify GPU (A100 preferred)
2. Install PhysicsNeMo + dependencies
3. Mount Google Drive and create checkpoint directories
4. Clone / pull the project repo
5. Smoke-test all critical imports

**Runtime:** GPU → A100 (Runtime → Change runtime type → A100)

## 1. GPU check

In [ ]:
import subprocess, sys

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if 'A100' in result.stdout:
        print('✓  A100 detected — optimal for PhysicsNeMo training.')
    elif 'T4' in result.stdout:
        print('⚠  T4 detected — training will work but slower.')
    else:
        print(result.stdout or 'GPU detected.')
except FileNotFoundError:
    print('No GPU detected — running on CPU.')
    print('This is fine for 00_env_setup and 01_data_pipeline.')
    print('Switch to A100 GPU before running training notebooks (Steps 5–7).')

In [ ]:
import torch

print(f'PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA {torch.version.cuda}  |  Device: {torch.cuda.get_device_name(0)}')
else:
    print('CUDA not available — CPU session confirmed.')

## 2. Install PhysicsNeMo

In [ ]:
if torch.cuda.is_available():
    cuda_ver = torch.version.cuda or ''
    extra = 'cu13' if cuda_ver.startswith('13') else 'cu12'
    print(f'GPU session — installing nvidia-physicsnemo[{extra},nn-extras] ...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', f'nvidia-physicsnemo[{extra},nn-extras]'], check=True)
else:
    print('CPU session — skipping PhysicsNeMo GPU install.')
    print('PhysicsNeMo will be installed with GPU extras when you switch to A100.')

print('Done.')

In [ ]:
# Install remaining project dependencies
!pip install -q h5py scipy botorch gpytorch fluidfoam trimesh numpy-stl tqdm pyyaml
!pip install -q huggingface-hub datasets torch-geometric
print('Dependencies installed.')

## 3. Mount Google Drive + checkpoint dirs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All training artefacts live here — persists across sessions
BASE_DIR = '/content/drive/MyDrive/cfd-ald-app'

DIRS = [
    f'{BASE_DIR}/checkpoints/flow_head',
    f'{BASE_DIR}/checkpoints/heat_head',
    f'{BASE_DIR}/checkpoints/species_head',
    f'{BASE_DIR}/checkpoints/multihead',
    f'{BASE_DIR}/data/processed/airfrans',
    f'{BASE_DIR}/data/processed/cfdbench',
    f'{BASE_DIR}/data/processed/showerhead_openfoam',
    f'{BASE_DIR}/logs',
]

for d in DIRS:
    os.makedirs(d, exist_ok=True)
    print(f'  ✓ {d}')

print('\nDrive mounted and checkpoint directories ready.')

## 4. Clone / pull project repo

In [ ]:
import os, sys

REPO_URL = 'https://github.com/Ranaam21/cfd-ald-app.git'
REPO_DIR = '/content/cfd-ald-app'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

# Add to Python path so all modules are importable
sys.path.insert(0, REPO_DIR)
print(f'Repo at {REPO_DIR} is on sys.path.')

## 5. Smoke-test imports

In [ ]:
import importlib, traceback

checks = [
    ('torch',                  lambda m: f'PyTorch {m.__version__}'),
    ('physicsnemo',            lambda m: f'PhysicsNeMo {m.__version__}'),
    ('physicsnemo.models',     lambda m: 'models OK'),
    ('h5py',                   lambda m: f'h5py {m.__version__}'),
    ('botorch',                lambda m: f'BoTorch {m.__version__}'),
    ('scipy',                  lambda m: f'SciPy {m.__version__}'),
    ('torch_geometric',        lambda m: f'PyG {m.__version__}'),
    ('physics.calculator',     lambda m: 'calculator OK'),
    ('physics.guardrails',     lambda m: 'guardrails OK'),
]

all_ok = True
for mod_name, desc_fn in checks:
    try:
        m = importlib.import_module(mod_name)
        print(f'  ✓  {desc_fn(m)}')
    except Exception:
        print(f'  ✗  {mod_name}')
        traceback.print_exc()
        all_ok = False

print()
print('Environment ready ✓' if all_ok else 'Some imports failed — check above.')

## 6. Quick physics calculator sanity check

In [ ]:
from physics.calculator import compute_all, k_rxn_from_sticking
from physics.guardrails import GuardrailEngine, GuardrailBounds

# Example: N2 carrier gas, 2mm nozzle, 10 m/s exit velocity, TMA precursor
params = {
    'rho':     1.12,      # kg/m³  (N2 at ~120°C, 1 atm)
    'V':       5.0,       # m/s    (mean nozzle exit velocity)
    'L':       0.002,     # m      (nozzle diameter 2 mm)
    'mu':      2.0e-5,    # Pa·s   (N2 at 120°C)
    'cp':      1040.0,    # J/kg·K (N2)
    'k_fluid': 0.031,     # W/m·K  (N2 at 120°C)
    'D_m':     2.5e-5,    # m²/s   (TMA in N2, ~120°C)
    'a':       380.0,     # m/s    (speed of sound N2 at 120°C)
    'k_rxn':   k_rxn_from_sticking(beta=0.05, v_th=144.0),
}

numbers = compute_all(params)
print('Dimensionless numbers for example ALD showerhead case:')
print('-' * 45)
for k, v in sorted(numbers.items()):
    print(f'  {k:6s} = {v:.4g}')

print()
engine = GuardrailEngine()
result = engine.check(numbers)
print(result.summary())

## 7. Convenience: save session config

Run this cell to write a `session_config.json` to Drive so other notebooks pick up the correct paths automatically.

In [ ]:
import json

session_cfg = {
    'base_dir':    BASE_DIR,
    'repo_dir':    REPO_DIR,
    'device':      'cuda' if torch.cuda.is_available() else 'cpu',
    'gpu_name':    torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
    'cuda_version': torch.version.cuda,
}

cfg_path = f'{BASE_DIR}/session_config.json'
with open(cfg_path, 'w') as f:
    json.dump(session_cfg, f, indent=2)

print(f'Session config saved to {cfg_path}')
print(json.dumps(session_cfg, indent=2))